In [1]:
pip install llama-index-core llama-index-graph-stores-neo4j llama-index-llms-openai

   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ------------------- -------------------- 5.8/11.9 MB 28.8 MB/s eta 0:00:01
   -------------------------------------- - 11.5/11.9 MB 30.5 MB/s eta 0:00:01
   ---------------------------------------- 11.9/11.9 MB 20.9 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 15.1 MB/s  0:00:00
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 39.6 MB/s  0:00:00
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   --------------------------------- ------ 6.0/7.2 MB 29.1 MB/s eta 0:00:01
   ---------------------------------------- 7.2/7.2 MB 27.4 MB/s  0:00:00
   ---------------------------------------- 0.0/818.2 kB ? eta -:--:--
   ---------------------------------------- 818.2/818.2 kB 34.5 MB/s  0:00:00
   ---------------------------------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import pandas as pd
import regex as re
from api.gpt_api import GPTClient
from deep_translator import GoogleTranslator
from tqdm import tqdm
from datetime import datetime, timezone
import math
import warnings

warnings.filterwarnings("ignore")

gpt_client = GPTClient()
deep_translator = GoogleTranslator(source='ja', target='en')

In [3]:
df = pd.read_csv("./data/exported-jira-data/Jira.csv")

def merge_name_and_ids(column_list, df):
        
    for column in column_list:
        cols = [col for col in df.columns if col.startswith(column)]
        
        col_pairs = []
        for col in cols:
            if "Id" not in col:
                col_split = col.split('.')
                if len(col_split) == 1:
                    id_col = f"{col} Id"
                else:
                    id_col = f"{col_split[0]} Id.{col_split[1]}"
                col_pairs.append((col, id_col))

        df[f"{column}"] = [

            [
                {
                    "name": row[n_col], 
                    "id": row[id_col]
                }

                for n_col, id_col in col_pairs
                if pd.notna(row[n_col])                
            ]

            for row in df.to_dict('records')

        ]
columns = ['Watchers', 'Reporter', 'Creator']
merge_name_and_ids(columns, df)
merged_columns = [col for col in df.columns if "merged" in col]
t_df = df[columns]
with pd.option_context(
    'display.max_rows', None,      # No row limit
    'display.max_columns', None,   # No column limit
    'display.max_colwidth', None,  # No cell content limit
    'display.width', 1000          # Prevent wrapping to a new line
):
    print(t_df.loc[0])

Watchers    [{'name': 'Farooqui, Ayaz (575) (EXT) [FUSO]', 'id': '712020:1e07aa0c-b20d-4386-b63d-f7f813f8f67d'}, {'name': 'Yoshihara, Yuji (575) [FUSO]', 'id': '712020:3bbdff6f-1498-435b-a25d-bf833b44b3f6'}]
Reporter                                                                                                        [{'name': 'Yoshihara, Yuji (575) [FUSO]', 'id': '712020:3bbdff6f-1498-435b-a25d-bf833b44b3f6'}]
Creator                                                                                                         [{'name': 'Yoshihara, Yuji (575) [FUSO]', 'id': '712020:3bbdff6f-1498-435b-a25d-bf833b44b3f6'}]
Name: 0, dtype: object


In [ ]:
print(df[merged_columns][0])

In [ ]:
df = pd.read_csv("./data/exported-jira-data/Jira.csv")
cols = [col for col in df.columns if col.startswith("Watchers")]

row = df.loc[0]

p_list = []
for col in cols:
    if not pd.isna(row[col]):
        p_dict = {}
        if "Id" not in col:
            col_split = col.split('.')
            if len(col_split) == 1:
                p_dict["name"] = row[col]
                p_dict["id"] = row[f"{col} Id"]
            else:
                p_dict["name"] = row[col]
                p_dict["id"] = row[f"{col_split[0]} Id.{col_split[1]}"]

            p_list.append(p_dict)

print(p_list)

[{'name': 'Farooqui, Ayaz (575) (EXT) [FUSO]', 'id': '712020:1e07aa0c-b20d-4386-b63d-f7f813f8f67d'}, {'name': 'Yoshihara, Yuji (575) [FUSO]', 'id': '712020:3bbdff6f-1498-435b-a25d-bf833b44b3f6'}]


In [ ]:
df = pd.read_csv('./preprocessed-exports/preprocessed_data_for_json.csv')
print(df['Issue Type'].value_counts())
print(df['Custom field (MFTBCFFR Department)'].value_counts())
print(df['Custom field (MFTBCFFR Bug Type)'].value_counts().sum())
print(df['Custom field (MFTBCFFR Track)'].value_counts().sum())
print(df['Custom field (MFTBCFFR FORCE System)'].value_counts())
print(df.columns)

In [9]:
# label_cols = [col for col in self.df.columns if col.startswith('Labels')]
# watcher_cols = [col for col in self.df.columns if col.startswith('Watchers')]
# impacted_fdp_cols = [col for col in self.df.columns if col.startswith('Custom field (MFTBCFFR Impacted FDPs)')]
# attachment_cols = [col for col in self.df.columns if col.startswith('Attachment')]
# comment_cols = [col for col in self.df.columns if col.startswith('Comment')]
# inward_issue_link_blocks_cols = [col for col in self.df.columns if col.startswith('Inward issue link (Blocks)')]
# outward_issue_link_blocks_cols = [col for col in self.df.columns if col.startswith('Outward issue link (Blocks)')]
# inward_issue_link_cloners_cols = [col for col in self.df.columns if col.startswith('Inward issue link (Cloners)')]
# outward_issue_link_cloners_cols = [col for col in self.df.columns if col.startswith('Outward issue link (Cloners)')]
# inward_issue_link_wbsgantt_cols = [col for col in self.df.columns if col.startswith('Inward issue link (Contains(WBSGantt))')]
# outward_issue_link_wbsgantt_cols = [col for col in self.df.columns if col.startswith('Outward issue link (Contains(WBSGantt))')]
# inward_issue_link_defect_cols = [col for col in self.df.columns if col.startswith('Inward issue link (Defect)')]
# outward_issue_link_defect_cols = [col for col in self.df.columns if col.startswith('Outward issue link (Defect)')]
# inward_issue_link_discovery_conn_cols = [col for col in self.df.columns if col.startswith('Inward issue link (Discovery - Connected)')]
# outward_issue_link_discovery_conn_cols = [col for col in self.df.columns if col.startswith('Outward issue link (Discovery - Connected)')]
# inward_issue_link_duplicate_cols = [col for col in self.df.columns if col.startswith('Inward issue link (Duplicate)')]
# outward_issue_link_duplicate_cols = [col for col in self.df.columns if col.startswith('Outward issue link (Duplicate)')]

class PreProcessor:
    def __init__(self, filepath='./data/exported-jira-data/Jira.csv'):
        self.filepath = Path(filepath)
        self.df = pd.read_csv(self.filepath, low_memory=False)
        self.df = self.df.dropna(axis=1, how='all')
        self.df_cols_intial = self.df.columns

    def get_df(self):
        return self.df

    def main_pipeline(self):


        self.filter_columns()

        #----------------- Processing people names -------------------#
        #-------------- Reporter || Creator || Watcher ---------------# 
        name_cols = [col for col in self.df.columns if col.startswith('Watchers') and "Id" not in col]
        name_col_list = ['Reporter','Creator']
        name_cols.extend(name_col_list)
        self.clean_names(name_cols)
        self.merge_name_and_ids(name_cols)
        self.drop_extra_columns(name_cols)

          
        #-------------------- Cleaning ---------------------------------#
        #----------------- Summary & Description -----------------------#
        text_cols = ['Summary','Description']      
        self.clean_texts(text_cols)


        #---------------- Fill Null Values ----------------------------#
        fillna_columns = ['Resolution']
        self.column_fillna(fillna_columns)

        #-------------------- Processing Date Columns -----------------#
        date_columns = ['Created', 'Updated', 'Resolved', 'Due date', 'Custom field ([CHART] Date of First Response)', 'Custom field (Incident Date)', 'Custom field (Closed Date)', 'Custom field (End Date)', 'Custom field (Closed Date)']
        self.convert_to_datetime(date_columns)

        self.normalize_status_in_time()

        label_cols = [col for col in self.df.columns if col.startswith('Labels')]
        #watcher_cols = [col for col in self.df.columns if col.startswith('Watchers')]
        impacted_fdp_cols = [col for col in self.df.columns if col.startswith('Custom field (MFTBCFFR Impacted FDPs)')]
        inward_issue_link_blocks_cols = [col for col in self.df.columns if col.startswith('Inward issue link (Blocks)')]
        outward_issue_link_blocks_cols = [col for col in self.df.columns if col.startswith('Outward issue link (Blocks)')]
        inward_issue_link_cloners_cols = [col for col in self.df.columns if col.startswith('Inward issue link (Cloners)')]
        outward_issue_link_cloners_cols = [col for col in self.df.columns if col.startswith('Outward issue link (Cloners)')]
        inward_issue_link_wbsgantt_cols = [col for col in self.df.columns if col.startswith('Inward issue link (Contains(WBSGantt))')]
        outward_issue_link_wbsgantt_cols = [col for col in self.df.columns if col.startswith('Outward issue link (Contains(WBSGantt))')]
        inward_issue_link_defect_cols = [col for col in self.df.columns if col.startswith('Inward issue link (Defect)')]
        outward_issue_link_defect_cols = [col for col in self.df.columns if col.startswith('Outward issue link (Defect)')]
        inward_issue_link_discovery_conn_cols = [col for col in self.df.columns if col.startswith('Inward issue link (Discovery - Connected)')]
        outward_issue_link_discovery_conn_cols = [col for col in self.df.columns if col.startswith('Outward issue link (Discovery - Connected)')]
        inward_issue_link_duplicate_cols = [col for col in self.df.columns if col.startswith('Inward issue link (Duplicate)')]
        outward_issue_link_duplicate_cols = [col for col in self.df.columns if col.startswith('Outward issue link (Duplicate)')]

        self.combine_columns(label_cols, 'Labels')
        #self.combine_columns(watcher_cols, 'Watchers')
        self.combine_columns(impacted_fdp_cols, 'Impacted Environments')
        self.combine_columns(inward_issue_link_blocks_cols, 'Inward issue link (Blocks)')
        self.combine_columns(outward_issue_link_blocks_cols, 'Outward issue link (Blocks)')
        self.combine_columns(inward_issue_link_cloners_cols, 'Inward issue link (Cloners)')
        self.combine_columns(outward_issue_link_cloners_cols, 'Outward issue link (Cloners)')
        self.combine_columns(inward_issue_link_wbsgantt_cols, 'Inward issue link (Contains(WBSGantt))')
        self.combine_columns(outward_issue_link_wbsgantt_cols, 'Outward issue link (Contains(WBSGantt))')
        self.combine_columns(inward_issue_link_defect_cols, 'Inward issue link (Defect)')
        self.combine_columns(outward_issue_link_defect_cols, 'Outward issue link (Defect)')
        self.combine_columns(inward_issue_link_discovery_conn_cols, 'Inward issue link (Discovery - Connected)')
        self.combine_columns(outward_issue_link_discovery_conn_cols, 'Outward issue link (Discovery - Connected)')
        self.combine_columns(inward_issue_link_duplicate_cols, 'Inward issue link (Duplicate)')
        self.combine_columns(outward_issue_link_duplicate_cols, 'Outward issue link (Duplicate)')

        #-------------------------- Processing Comments -----------------------------#
        comment_cols = [col for col in self.df.columns if col.startswith('Comment')]
        self.process_comments(comment_cols)
        self.combine_columns(comment_cols, 'Comments')
        #self.translate_to_enu('Summary')
        self.export_as_csv()

        return self.df

    def drop_extra_columns(self, col_names):
        for col in col_names:
            cols = [column for column in self.df.columns if column.startswith(col) and column != col]
            self.df = self.df.drop(columns=cols, errors='ignore')

    def filter_columns(self):
        # No dropping 'Reporter Id', 'Creator Id' columns
        columns_to_drop = ['Project key', 'Project name', 'Project type', 'Project lead', 'Project lead id', 'Assignee', 'Assignee Id', 'Last Viewed', 'Environment', 'Votes', 'Custom field (Approvals)', 'Custom field (Begin Date)', 'Custom field (End Date (migrated 2))', 'Custom field (Epic Color)', 'Custom field (Issue color)', 'Custom field (MFTBCDLS Request Type)', 'Custom field (Start Date (migrated 2))', 'Custom field (TRKD Region)', 'Custom field (Rank)']
        attachment_columns_to_drop = [col for col in self.df_cols_intial if col.startswith('Attachment')]
        #watchers_id_columns_to_drop = [col for col in self.df_cols_intial if col.startswith('Watchers Id')]
        self.df = self.df.drop(columns=columns_to_drop, errors='ignore')
        self.df = self.df.drop(columns=attachment_columns_to_drop, errors='ignore')
        #self.df = self.df.drop(columns=watchers_id_columns_to_drop, errors='ignore')


    def merge_name_and_ids(self, column_list):
        
        for column in column_list:
            cols = [col for col in self.df.columns if col.startswith(column)]
            
            col_pairs = []
            for col in cols:
                if "Id" not in col:
                    col_split = col.split('.')
                    if len(col_split) == 1:
                        id_col = f"{col} Id"
                    else:
                        id_col = f"{col_split[0]} Id.{col_split[1]}"
                    col_pairs.append((col, id_col))

            self.df[f"{column}"] = [

                [
                    {
                        "name": row[n_col], 
                        "id": row[id_col]
                    }

                    for n_col, id_col in col_pairs
                    if pd.notna(row[n_col])                
                ]

                for row in self.df.to_dict('records')

            ]

                
    def column_fillna(self, col_list):
        for col in col_list:
            match col:
                case 'Resolution':
                    self.df[col] = self.df[col].fillna('Unresolved')

                case 'Custom field (Epic Name)':
                    self.df[col] = self.df[col].fillna('Not Applicable (Not an EPIC)')

                case 'Custom field (Epic Status)':
                    self.df[col] = self.df[col].fillna('Not Applicable (Not an EPIC)')

    def clean_names(self, col_list):

        def clean_name(x):
            match = re.search("[a-zA-Z,\s]+(?=\s*\()", str(x))
            if match is None:
                return x
            x = match.group()
            x_l = x.split(',')
            for l in x_l:
                l = str(l).strip()
            x_l[0], x_l[1] = x_l[1], x_l[0]
            x = "".join(x_l)
            x = x.strip()
            return x

        for col in col_list:
            self.df[col] = self.df[col].apply(clean_name)

    def clean_texts(self, col_list):
    
        def clean_text(text):
            text = str(text)
            pattern_img = r"(?i)(?:!image)?.*?!"
            pattern_sq_bracket = r"\[[^\]]*\]"
            text = re.sub(pattern_img, "", text)
            text = re.sub(pattern_sq_bracket, "", text)
            text = text.replace("\r\n", "\n")
            text = text.replace("\r", "\n")
            text = re.sub(r"[ \t]+", " ", text)
            text = re.sub(r"\n{2,}", "\n", text)
            text = text.strip()
            return text

        for col in col_list:
            self.df[col] = self.df[col].apply(clean_text)

    def convert_to_datetime(self, col_list):

        def to_isoformat(x):
            return x.isoformat()

        for col in col_list:
            self.df[col] = pd.to_datetime(self.df[col])
            self.df[col] = self.df[col].apply(to_isoformat)

    def normalize_status_in_time(self):

        status_id_map = {
            "3" : "In Progress",
            "4" : "Reopened",
            "6" : "Closed",
            "10035" : "To Do",
            "10036" : "Done",
            "10037" : "Active",
            "10038" : "InActive",
            "10039" : "On Hold",
            "10040" : "Under Review",
            "10041" : "In Testing",
            "10053" : "Reject Accepted",
            "10054" : "Rejected",
            "10055" : "Fixed",
            "10056" : "Validated",
            "10057" : "Specification Change"
        }
        #3_*:*_1_*:*_497400097_*|*_10035_*:*_1_*:*_1141757_*|*_10036_*:*_1_*:*_0
        def convert_to_json(x):
            if x is None or (isinstance(x, float) and math.isnan(x)):
                return None
            stat_list = []
            time_blocks = str(x).split('_*|*_')
            for block in time_blocks:
                stats = block.split('_*:*_')
                s = {}
                s['status'] = status_id_map[str(stats[0])]
                s['duration_in_hrs'] = round(float(stats[2]) / 1000 / 3600, 2)
                s['duration_in_days'] = round(float(stats[2]) / 1000 / 3600 / 24, 2)
                stat_list.append(s)
            return stat_list

        self.df['Custom field ([CHART] Time in Status)'] = self.df['Custom field ([CHART] Time in Status)'].apply(convert_to_json)

    def combine_columns(self, col_list, new_col_name):

        def build_list(row):
            return [row[col] for col in col_list if not pd.isna(row[col])]
            
        self.df[new_col_name] = self.df.apply(build_list, axis=1)
        col_list = [col for col in col_list if col != new_col_name]
        self.df = self.df.drop(columns = col_list, errors='coerce')     

    def process_comments(self, col_list):

        def find_author(x):
            if pd.isna(x) or not isinstance(x, str):
                return None

            pattern = r"(?i)(?:(?:best\s+)?regards?|thanks?)\s*,\s*(.*)$"
            match = re.search(pattern, x, re.DOTALL)
            author = None
            if match:
                author = match.group(1)
                author = author.replace('\r',' ').replace('\n',' ')
                author = author.strip()
                return author

        def clean_comments(x):

            pattern = r"(?i)(?:dear|hi)?\s*\[[^\]]*\]\s*(?:san)?\s*,?\s*"
            pattern_img = r"(?i)(?:!image)?.*?!"
            pattern_sq_bracket = r"\[[^\]]*\]"
            text = re.sub(pattern, "", x, count=1)
            text = re.sub(pattern_img, "", text)
            text = re.sub(pattern_sq_bracket, "", text)
            text = text.replace("\r\n", "\n")
            text = text.replace("\r", "\n")
            text = re.sub(r"[ \t]+", " ", text)
            text = re.sub(r"\n{2,}", "\n", text)
            text = text.strip()
            return text

        def convert_to_json(x):
            if x is None or (isinstance(x,float) and math.isnan(x)):
                return None
            data_list = str(x).split(";")
            timestamp = pd.to_datetime(data_list[0]).isoformat()
            author = find_author(data_list[2])
            text = clean_comments(data_list[2])
            return {
                "timestamp" : timestamp,
                "author" : author,
                "text" : text
            }

        for col in col_list:
            self.df[col] = self.df[col].apply(convert_to_json)

    def translate_to_enu(self, col):

        def translate(x):
            x = str(x).strip()
            jp_regex = r'[\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FAF]'
            if bool(re.search(jp_regex, str(x))):
                return deep_translator.translate(str(x)) if pd.notna(x) and str(x).strip() else ""
            return x
    
        new_col_name = f"{col}_enu"
        self.df[new_col_name] = self.df[col].apply(translate)

    def export_as_csv(self, filepath = './data/preprocessed-exports', filename = 'preprocessed_data_people_with_ids.csv'):
        filepath = Path(filepath)
        filepath.mkdir(exist_ok=True)
        self.df.to_csv(filepath/filename, index=False)
        #self.df.to_csv(filepath/'preprocessed_data_for_trans.csv', index=False)


preprocessor = PreProcessor()
df = preprocessor.main_pipeline()
print(df.shape, '\n')
#print(df['Summary'], '\n')
#print(df['Resolution'].value_counts(), '\n')
#print(df['Watchers'].value_counts(), '\n')
#print(df['Labels'].value_counts(), '\n')
#print(df['Created'].value_counts(), '\n')
#print(df['Custom field (Epic Status)'].value_counts(), '\n')
#print(df['Custom field ([CHART] Time in Status)'].value_counts(), '\n')
#print(df['Created'].dtype, '\n')
#print(df['Labels'].value_counts(), '\n')


(5807, 56) 



In [10]:
print(df.columns)

Index(['Summary', 'Issue key', 'Issue id', 'Issue Type', 'Status', 'Priority',
       'Resolution', 'Reporter', 'Creator', 'Created', 'Updated', 'Resolved',
       'Due date', 'Labels', 'Description', 'Watchers',
       'Inward issue link (Blocks)', 'Outward issue link (Blocks)',
       'Inward issue link (Cloners)', 'Outward issue link (Cloners)',
       'Inward issue link (Contains(WBSGantt))',
       'Outward issue link (Contains(WBSGantt))', 'Inward issue link (Defect)',
       'Outward issue link (Defect)',
       'Inward issue link (Discovery - Connected)',
       'Outward issue link (Discovery - Connected)',
       'Inward issue link (Duplicate)', 'Outward issue link (Duplicate)',
       'Custom field (Closed Date)', 'Custom field (End Date)',
       'Custom field (Epic Name)', 'Custom field (Epic Status)',
       'Custom field (FORCE Root Cause Analysis)',
       'Custom field (Incident Date)', 'Custom field (MFTBCFFR Action Taken)',
       'Custom field (MFTBCFFR Bug Category)

In [ ]:
class PreProcessorTrans:
    def __init__(self, filepath = './preprocessed-exports/preprocessed_data.csv'):
        self.filepath = Path(filepath)
        self.df = pd.read_csv(self.filepath)

    def translate_in_batches_recur(self, start, end, col, batch_size=200, filepath = './preprocessed-exports/preprocessed_data_for_trans.csv'):
        temp_df = pd.read_csv(filepath)
        num_rows = temp_df.shape[0]

        for i in range(start, end):
            print(f"Processing Index {i}")
            if i == num_rows:
                return
            x = str(temp_df.loc[i,col]).strip()
            jp_regex = r'[\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FAF]'
            try:
                if bool(re.search(jp_regex, str(x))):
                    temp_df.loc[i,f"{col}_enu"] = deep_translator.translate(str(x)) if pd.notna(x) and str(x).strip() else ""
            except:
                return

        temp_df.to_csv(filepath, index=False)

        self.translate_in_batches(start+batch_size, end+batch_size, col)

    def translate_in_batches_no_recur(self, start, col, filepath = './preprocessed-exports/preprocessed_data_for_trans.csv'):
            temp_df = pd.read_csv(filepath)
            num_rows = temp_df.shape[0]
    
            for i in range(start, num_rows):
                print(f"Processing Index {i}")
                x = str(temp_df.loc[i,col]).strip()
                jp_regex = r'[\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FAF]'
                try:
                    if bool(re.search(jp_regex, str(x))):
                        temp_df.loc[i,f"{col}_enu"] = deep_translator.translate(str(x)) if pd.notna(x) and str(x).strip() else ""
                except:
                    return
                if i%200 == 0:
                    temp_df.to_csv(filepath, index=False)

    def clean_data(self, filepath = './preprocessed-exports/preprocessed_data_for_trans.csv'):
        temp_df = pd.read_csv(filepath)
        num_rows = temp_df.shape[0]
        col_list = [col for col in temp_df.columns if col.startswith('Unnamed')]
        temp_df = temp_df.drop(columns=col_list)
        temp_df.to_csv(filepath, index=False)
        print(col_list)
        print(num_rows)
        print(temp_df.head())


translate_engine = PreProcessorTrans()
translate_engine.translate_in_batches_no_recur(0, 'Summary')
        


In [10]:
print(deep_translator.translate("CMVM_顧客住所が違う"))
print(deep_translator.translate("VO_リサイクルステータス更新_中国"))

CMVM_Customer address is different
VO_Recycling status update_China


In [125]:
print(df.loc[2000,'Comments'])

[{'timestamp': '2025-05-29T04:41:00', 'author': 'Akash', 'text': 'Thank you for your support.\nWe have acknowledged this SR and we are working on it. We will provide you an update soon.\nMeanwhile, please let us know if you have any query.\nBest Regards,\nAkash'}, {'timestamp': '2025-05-29T07:18:00', 'author': 'Soham', 'text': 'Thank you for your support.\nThe recovery is completed @Okayama-PROD.\nPlease check and verify.\nPlease let us know in case of an query.\nBest Regards,\nSoham'}]


NameError: name 'pd' is not defined

In [119]:
def find_author(x):
    if pd.isna(x) or not isinstance(x, str):
        return None

    pattern = r"(?i)(?:(?:best\s+)?regards?|thanks?)\s*,\s*(.*)$"
    match = re.search(pattern, x, re.DOTALL)
    author = None
    if match:
        author = match.group(1)
        author = author.replace('\r',' ').replace('\n',' ')
        author = author.strip()
        return author

def remove_account_id(x):

    pattern = r"(?i)(?:dear|hi)?\s*\[[^\]]*\]\s*(?:san)?\s*,?\s*"
    pattern_img = r"(?i)(?:!image)?.*?!"
    pattern_sq_bracket = r"\[[^\]]*\]"
    text = re.sub(pattern, "", x, count=1)
    text = re.sub(pattern_img, "", text)
    text = re.sub(pattern_sq_bracket, "", text)
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    text = text.strip()
    return text

def convert_to_json(x):
    data_list = x.split(";")
    timestamp = pd.to_datetime(data_list[0]).isoformat()
    author = find_author(data_list[2])
    text = remove_account_id(data_list[2])
    return {
        "timestamp" : timestamp,
        "author" : author,
        "text" : text
    }

text = "22/Oct/25 11:20 AM;712020:37fe73dd-d299-42bc-841b-6a62e095b6ee;Dear [~accountid:712020:66b35490-1d18-4551-8edc-a61d8fcec8ca] San,\r\n\r\nThanks for your support.\r\n\r\nWe have acknowledged this SR and working on it. We'll provide an update soon.\r\nMeanwhile, please let us know in case of any query.\r\n\r\nBest Regards,\r\nHimanshu"
text2 = '22/Jul/26 4:46 PM;712020:234ec49a-8166-4fec-8b0f-cb8571c348ca;Dear [~accountid:712020:da8caa97-eb32-49fd-9437-1fff5327f422] San,\n\nThank you for your support.\n\nWe have migrated the Job Card and the Activity for the Sales Order: A042368030.\n\n!image-20260722-164449.png|width=462,alt="image-20260722-164449.png"!\n\n!image-20260722-164544.png|width=447,alt="image-20260722-164544.png"!\n\nThanks,\nNandini'
print(convert_to_json(text2))

{'timestamp': '2026-07-22T16:46:00', 'author': 'Nandini', 'text': 'Thank you for your support.\nWe have migrated the Job Card and the Activity for the Sales Order: A042368030.\nThanks,\nNandini'}


In [ ]:
print(df['Comment'].value_counts(), '\n')

In [28]:
print(df[df['Issue key'] == 'MFTBCFFR-6081']['Custom field ([CHART] Time in Status)'].iloc[0])

None


In [44]:
def clean_watchers(x):
    match = re.search("[a-zA-Z,\s]+(?=\s*\()", str(x))
    if match is None:
        return x
    x = match.group()
    x_l = x.split(',')
    for l in x_l:
        l = str(l).strip()
    x_l[0], x_l[1] = x_l[1], x_l[0]
    x = "".join(x_l)
    return x

watcher = df['Watchers'].apply(clean_watchers)
print(watcher[0])
#watchers = watchers.apply(clean_watchers)
#print(watchers.loc[0])

# for w in watcher:
#     count = 0
#     match = re.search("[a-zA-Z,\s]+(?=\s*\()", str(w))
#     if match is not None:
#         print(match.group())


 Ayaz Farooqui


<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
C:\Users\SaadAhmed\AppData\Local\Temp\ipykernel_10112\2185598186.py:2: SyntaxWarning: invalid escape sequence '\s'
  match = re.search("[a-zA-Z,\s]+(?=\s*\()", str(x))


In [6]:
filepath = Path("./exported-jira-data/Jira.csv")
jira_df = pd.read_csv(filepath, low_memory=False)
jira_df_cleaned = jira_df.dropna(axis=1, how='all')
jira_df_cleaned = jira_df_cleaned.drop(columns=['Project key', 'Project name', 'Project type', 'Project lead', 'Project lead id', 'Assignee', 'Assignee Id', 'Reporter Id', 'Creator Id', 'Last Viewed', 'Environment', 'Votes',  'Custom field (Approvals)', 'Custom field (Begin Date)', 'Custom field (End Date (migrated 2))', 'Custom field (Epic Color)', 'Custom field (Issue color)', 'Custom field (MFTBCDLS Request Type)', 'Custom field (Start Date (migrated 2))', 'Custom field (TRKD Region)'], errors='coerce')
print(jira_df_cleaned.shape)

jira_df_cleaned['Watchers Id']


(5807, 337)


0       712020:1e07aa0c-b20d-4386-b63d-f7f813f8f67d
1                          63d377128c3018ca8a1c6fa6
2       712020:66b35490-1d18-4551-8edc-a61d8fcec8ca
3       712020:66b35490-1d18-4551-8edc-a61d8fcec8ca
4       712020:1e07aa0c-b20d-4386-b63d-f7f813f8f67d
                           ...                     
5802    712020:3e127392-09a7-4e65-998e-e645047840d7
5803    712020:3e127392-09a7-4e65-998e-e645047840d7
5804    712020:3e127392-09a7-4e65-998e-e645047840d7
5805    712020:3e127392-09a7-4e65-998e-e645047840d7
5806    712020:3e127392-09a7-4e65-998e-e645047840d7
Name: Watchers Id, Length: 5807, dtype: str

In [5]:
for cols in jira_df_cleaned.columns:
    print(cols)

Summary
Issue key
Issue id
Issue Type
Status
Priority
Resolution
Reporter
Creator
Created
Updated
Resolved
Due date
Labels
Labels.1
Labels.2
Labels.3
Description
Watchers
Watchers.1
Watchers.2
Watchers.3
Watchers.4
Watchers.5
Watchers.6
Watchers.7
Watchers.8
Watchers.9
Watchers Id
Watchers Id.1
Watchers Id.2
Watchers Id.3
Watchers Id.4
Watchers Id.5
Watchers Id.6
Watchers Id.7
Watchers Id.8
Watchers Id.9
Inward issue link (Blocks)
Outward issue link (Blocks)
Outward issue link (Blocks).1
Inward issue link (Cloners)
Inward issue link (Cloners).1
Inward issue link (Cloners).2
Inward issue link (Cloners).3
Inward issue link (Cloners).4
Inward issue link (Cloners).5
Outward issue link (Cloners)
Inward issue link (Contains(WBSGantt))
Inward issue link (Contains(WBSGantt)).1
Outward issue link (Contains(WBSGantt))
Outward issue link (Contains(WBSGantt)).1
Outward issue link (Contains(WBSGantt)).2
Outward issue link (Contains(WBSGantt)).3
Inward issue link (Defect)
Inward issue link (Defect).

In [ ]:
# Sprint
# Custom field (Start Date (migrated 2)) - Remove
# Custom field (TRKD Region) - Remove
# Custom field ([CHART] Date of First Response)
# Custom field ([CHART] Time in Status)
print(jira_df_cleaned['Created'].value_counts(), '\n')
print(jira_df_cleaned['Custom field (Start Date (migrated 2))'].value_counts().sum(), '\n')
print(jira_df_cleaned['Custom field (End Date (migrated 2))'].value_counts(), '\n')


In [ ]:
t_df = jira_df_cleaned[jira_df_cleaned['Custom field (Epic Name)'].notna()]
print(t_df.shape)
print(t_df.head())
print(jira_df_cleaned['Custom field (Epic Name)'].value_counts())
print(jira_df_cleaned['Custom field (Epic Status)'].value_counts())
print(jira_df_cleaned['Priority'].value_counts())
print(jira_df_cleaned['Custom field (Severity)'].value_counts())
print(jira_df_cleaned['Status Category Changed'].value_counts())
print(jira_df_cleaned['Custom field ([CHART] Time in Status)'].value_counts())


## Columns to Drop

- Project key
- Project name
- Project type
- Project lead
- Project lead id
- Assignee
- Assignee Id
- Reporter Id
- Creator Id
- Last Viewed
- Environment
- Votes
- Attachment Columns
- Custom field (Approvals)
- Custom field (Begin Date)
- Custom field (End Date (migrated 2))
- Custom field (Epic Color)
- Custom field (Issue color)
- Custom field (MFTBCDLS Request Type)
- Custom field (Start Date (migrated 2))
- Custom field (TRKD Region)

## Important Fields to keep

- Summary (Translated)
- Issue Key
- Issue Type 
- Status 
- Priority
- Resolution
- Labels (Combined)
- Description (Translated)
- Custom field (Epic Name)
- Custom field (Epic Status)
- Custom field (FORCE Root Cause Analysis) 
- Custom field (MFTBCFFR Action Taken)
- Custom field (MFTBCFFR Bug Category)
- Custom field (MFTBCFFR Bug Type)
- Custom field (MFTBCFFR Business Area)
- Custom field (MFTBCFFR CR Phase)
- Custom field (MFTBCFFR Department)
- Custom field (MFTBCFFR FORCE System)
- Custom field (MFTBCFFR Impacted FDPs) (Combined)
- Custom field (MFTBCFFR Issue Raised FDP)
- Custom field (MFTBCFFR RCA Category)
- Custom field (MFTBCFFR Sub Category)
- Custom field (MFTBCFFR Track)
- Custom field (Severity)
- Sprint
- Parent
- Parent key
- Parent summary
- Status Category
- Status Category Changed

#### People

- Reporter
- Watchers (Combined)

#### Dates

- Created
- Updated (For those not yet closed done or marked)
- Resolved (For those closed or done)
- Due date
- Custom field ([CHART] Date of First Response)
- Custom field (Incident Date)
- Custom field (Closed Date)
- Custom field (End Date)

#### Special formatting required

- Custom field ([CHART] Time in Status)

#### Issue Links

- Inward issue link (Blocks) (Combined)
- Outward issue link (Blocks) (Combined)
- Inward issue link (Cloners) (Combined)
- Outward issue link (Cloners) (Combined)
- Inward issue link (Contains(WBSGantt)) (Combined)
- Outward issue link (Contains(WBSGantt)) (Combined)
- Inward issue link (Defect) (Combined)
- Outward issue link (Defect) (Combined)
- Inward issue link (Discovery - Connected) (Combined)
- Outward issue link (Discovery - Connected) (Combined)
- Inward issue link (Duplicate) (Combined)
- Outward issue link (Duplicate) (Combined)



In [ ]:
# Inward issue link (Blocks)
# Outward issue link (Blocks)
# Inward issue link (Cloners)
# Outward issue link (Cloners)
# Inward issue link (Contains(WBSGantt))
# Outward issue link (Contains(WBSGantt))
# Inward issue link (Defect)
# Outward issue link (Defect)
# Inward issue link (Discovery - Connected)
# Outward issue link (Discovery - Connected)
# Inward issue link (Duplicate)
# Outward issue link (Duplicate)

print(jira_df_cleaned['Inward issue link (Blocks)'].value_counts())
print(jira_df_cleaned['Outward issue link (Blocks)'].value_counts())
print(jira_df_cleaned['Inward issue link (Cloners)'].value_counts())
print(jira_df_cleaned['Outward issue link (Cloners)'].value_counts())
print(jira_df_cleaned['Inward issue link (Contains(WBSGantt))'].value_counts())
print(jira_df_cleaned['Outward issue link (Contains(WBSGantt))'].value_counts())
print(jira_df_cleaned['Inward issue link (Defect)'].value_counts())
print(jira_df_cleaned['Outward issue link (Defect)'].value_counts())
print(jira_df_cleaned['Inward issue link (Discovery - Connected)'].value_counts())
print(jira_df_cleaned['Outward issue link (Discovery - Connected)'].value_counts())
print(jira_df_cleaned['Inward issue link (Duplicate)'].value_counts())
print(jira_df_cleaned['Outward issue link (Duplicate)'].value_counts())

In [ ]:
summaries = jira_df_cleaned['Summary'].str.strip()

tqdm.pandas(desc="Translating Jira Summaries")

jira_df_cleaned['Translated'] = jira_df_cleaned['Summary'].progress_apply(
    lambda x: deep_translator.translate(str(x)) if pd.notna(x) and str(x).strip() else ""
)
